In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/gdrive')
import os

# Try to read the CSV file. If the previous `ls` command shows the file, uncomment and use the correct path.
folder_path = '/content/gdrive/MyDrive/Time series Projet/data/separted'
path_x_train = f'{folder_path}/X_train.csv'
path_y_train = f'{folder_path}/y_train.csv'
path_x_test = f'{folder_path}/X_test.csv'
path_y_test = f'{folder_path}/y_test.csv'
df_x_train = pd.read_csv(path_x_train)
df_y_train = pd.read_csv(path_y_train)
df_x_test = pd.read_csv(path_x_test)
df_y_test = pd.read_csv(path_y_test)
df = pd.read_csv('/content/gdrive/MyDrive/Time series Projet/data/merged_tourism_data_final.csv')
df=pd.read_csv('/content/gdrive/MyDrive/Time series Projet/data/merged_tourism_data_final.csv')


Mounted at /content/gdrive


In [2]:
!pip install scikeras scikit-learn==1.5.2 optuna tensorflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 18.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


In [3]:
# 1. Préparation des données sans leakage (Scaling dynamique)
import optuna
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Dropout, Conv1D, MaxPooling1D, Flatten, Input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import TimeSeriesSplit

def prepare_data_wf(df_in, window_size=12):
    df_p = df_in.copy()
    features = [c for c in ['Arrivals', 'Nights', 'is_covid', 'InterTourismeReceipts', 'REER', 'Oil_price', 'FDI', 'Poverty_rate'] if c in df_p.columns and not df_p[c].isnull().all()]
    df_p[features] = df_p[features].interpolate().bfill().ffill()
    data = df_p[features].values
    
    # NE PAS SCALER ICI POUR ÉVITER LE LEAKAGE
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size, :])
        y.append(data[i+window_size, 0])
        
    X, y = np.array(X), np.array(y)
    train_size = int(len(X) * 0.8)
    return X, y, train_size

X_all, y_all, train_size = prepare_data_wf(df)
X_train_init, y_train_init = X_all[:train_size], y_all[:train_size]


[I 2026-05-13 10:38:12,211] A new study created in memory with name: no-name-608eca0b-1f3e-4bfe-b6d3-2cc1c4f9f3e3
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
[I 2026-05-13 10:38:20,500] Trial 0 finished with value: 0.02231421324377346 and parameters: {'model_type': 'GRU', 'units': 99, 'dropout': 0.25179001679100244, 'lr': 0.009875383764106028}. Best is trial 0 with value: 0.02231421324377346.
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
[I 2026-05-13 10:38:37,144] Trial 1 finished with value: 0.02316014819564119


Meilleure architecture: LSTM
Meilleurs hyperparamètres: {'model_type': 'LSTM', 'units': 126, 'dropout': 0.16029899336263173, 'lr': 0.002098914540310631}
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step

Résultats Finaux (LSTM):
R2 Score: 0.0909
RMSE: 0.2308


In [21]:
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    # Feed Forward Part
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x + res

def build_transformer_model(input_shape, head_size=64, num_heads=4, ff_dim=64, num_transformer_blocks=2, mlp_units=[64], dropout=0.1, mlp_dropout=0.1):
    inputs = layers.Input(shape=input_shape)
    x = inputs
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.GlobalAveragePooling1D(data_format="channels_last")(x)
    for dim in mlp_units:
        x = layers.Dense(dim, activation="relu")(x)
        x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

# Instantiate and train
input_shape = (X_train.shape[1], X_train.shape[2])
transformer_model = build_transformer_model(input_shape)
transformer_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="mse")

print("Training Transformer model...")
history = transformer_model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.1, verbose=0)

# Evaluation
y_pred_trans = transformer_model.predict(X_test)
print(f"\nTransformer Results:")
print(f"R2 Score: {r2_score(y_test, y_pred_trans):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_trans)):.4f}")

Training Transformer model...
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step

Transformer Results:
R2 Score: 0.2022
RMSE: 0.2162
